## Problem: Quantize Your Language Model

### Problem Statement
Implement a **language model** using an LSTM and apply **dynamic quantization** to optimize it for inference. Dynamic quantization reduces the model size and enhances inference speed by quantizing the weights of the model.

### Requirements

1. **Define the Language Model**:
   - **Components**:
     - **Embedding Layer**: Converts input tokens into dense vector representations.
     - **LSTM Layer**: Processes the embedded sequence to capture temporal dependencies.
     - **Fully Connected Layer**: Outputs predictions for the next token.
     - **Softmax Layer**: Applies a probability distribution over the vocabulary for predictions.
   - **Forward Pass**:
     - Pass the input sequence through the embedding layer.
     - Feed the embedded sequence into the LSTM.
     - Use the final hidden state from the LSTM to make predictions via the fully connected layer.
     - Apply the softmax function to obtain probabilities over the vocabulary.

2. **Apply Dynamic Quantization**:
   - Quantize the model
   - Evaluate the quantized model's performance compared to the original model.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.ao.quantization import quantize_dynamic

# Dynamic quantization needs a quantized backend. On Apple Silicon (and some other
# builds) the default engine is 'none', so pick one this torch build supports.
if torch.backends.quantized.engine == "none":
    engines = [e for e in torch.backends.quantized.supported_engines if e != "none"]
    if not engines:
        raise RuntimeError("this torch build has no quantized backend (fbgemm/qnnpack)")
    torch.backends.quantized.engine = engines[0]
print(f"Quantized engine: {torch.backends.quantized.engine}")

In [4]:
# Define a simple Language Model (e.g., an LSTM-based model)
class LanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super(LanguageModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        output = self.fc(lstm_out[:, -1, :])  # Use the last hidden state for prediction
        return self.softmax(output)

In [ ]:
# Create synthetic training data
torch.manual_seed(42)
vocab_size = 50
seq_length = 10
batch_size = 32
X_train = torch.randint(0, vocab_size, (batch_size, seq_length))  # Random integer input
y_train = torch.randint(0, vocab_size, (batch_size,))  # Random target words

# Initialize the model, loss function, and optimizer
embed_size = 64
hidden_size = 128
num_layers = 2
model = LanguageModel(vocab_size, embed_size, hidden_size, num_layers)

# The model ends in a softmax, so it returns probabilities, not logits.
# nn.CrossEntropyLoss would apply a second softmax on top and the loss would barely
# move, so train with NLLLoss on the log-probabilities instead.
criterion = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Training loop
epochs = 5
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    output = model(X_train)
    loss = criterion(torch.log(output.clamp_min(1e-9)), y_train)
    loss.backward()
    optimizer.step()

    # Log progress every epoch
    print(f"Epoch [{epoch + 1}/{epochs}] - Loss: {loss.item():.4f}")

# Now, we will quantize the model dynamically to reduce its size and improve inference speed
# Quantization: Apply dynamic quantization to the language model
quantized_model = quantize_dynamic(model, {nn.Linear, nn.LSTM}, dtype=torch.qint8)

# Save the float weights. A dynamically quantized state_dict stores the LSTM's packed
# weights as torch.ScriptObject, which torch.load(weights_only=True) (the default
# since PyTorch 2.6) refuses to unpickle. Dynamic quantization is deterministic, so
# quantizing the float weights again after loading rebuilds the same model.
torch.save(model.state_dict(), "language_model_fp32.pth")


In [ ]:
# Load the float weights, then apply the same dynamic quantization again
quantized_model = LanguageModel(vocab_size, embed_size, hidden_size, num_layers)
quantized_model.load_state_dict(torch.load("language_model_fp32.pth", weights_only=True))
quantized_model = quantize_dynamic(quantized_model, {nn.Linear, nn.LSTM}, dtype=torch.qint8)

In [ ]:
# Testing the quantized model on a sample input
quantized_model.eval()
test_input = torch.randint(0, vocab_size, (1, seq_length))
with torch.no_grad():
    prediction = quantized_model(test_input)
    print(f"Prediction for input {test_input.tolist()}: {prediction.argmax(dim=1).item()}")

In [ ]:
# Compare the quantized model with the float one: size of the saved weights and
# whether the two agree on the next-token prediction for the same inputs
import io

def state_dict_mb(m):
    buf = io.BytesIO()
    torch.save(m.state_dict(), buf)
    return buf.getbuffer().nbytes / 1e6

model.eval()
eval_input = torch.randint(0, vocab_size, (256, seq_length))
with torch.no_grad():
    fp32_probs = model(eval_input)
    int8_probs = quantized_model(eval_input)

agreement = (fp32_probs.argmax(dim=1) == int8_probs.argmax(dim=1)).float().mean().item()
print(f"Weights: {state_dict_mb(model):.2f} MB (fp32) -> {state_dict_mb(quantized_model):.2f} MB (int8)")
print(f"Top-1 agreement with the float model: {agreement:.1%}")
print(f"Max probability difference: {(fp32_probs - int8_probs).abs().max().item():.2e}")